# Here we try to exclude protein

In [2]:
from pmda.parallel import ParallelAnalysisBase
import numpy as np
# from radii import types_radii
from MDAnalysis import Universe
import MDAnalysis as mda
import warnings
import os
warnings.filterwarnings("ignore")

traj = 'GRO'

types_radii = {
'HGA1': 1.3200,
'HGA2': 1.3400,
'HOL': 0.2245,
'HAL1': 1.3200,
'HAL2': 1.3400,
'HAL3': 1.3400,
'HBL':  1.3200,
'HCL':  0.2245,
'HL':   0.7,
'HEL1': 1.25,
'HEL2': 1.26,
'CL':   2.00,
'CCL':  2.00,
'CTL1': 2.275,
'CTL2': 2.010,
'CTL3': 2.040,
'CTL5': 2.06,
'CEL1': 2.09,
'CEL2': 2.08,
'CRL1': 2.010,
'CRL2': 2.020,
'OBL':  1.70,
'OCL':  1.70,
'O2L':  1.70,
'OHL':  1.77,
'OSL':  1.65006,
'OSLP': 1.65006,
'NH3L': 1.85,
'NTL':  1.85,
'SL':   2.1,
'PL':   2.15,
'CC301':  2.000,
'CC3062': 2.000,
'CC311':  2.000,
'CC3161': 2.000,
'CC3162': 2.000,
'CC3163': 2.000,
'CC321':  2.010,
'CC3261': 2.010,
'CC3263': 2.010,
'CC331':  2.040,
'CC2O1':  2.000,
'CC2O2':  2.000,
'CC2O3':  2.000,
'CC2O4':  1.800,
'CC2O5':  1.700,
'OC2D1':  1.700,
'OC2D2':  1.700,
'OC2D3':  1.700,
'OC2D4':  1.700,
'OC3C61': 1.650,
'OC311':  1.765,
'NC2D1':  1.850,
'HCP1':   0.2245,
'HCA1':   1.340,
'HCA2':   1.340,
'HCA3':   1.340,
'OC2DP':  1.70,
'OC312':  1.77,
'OC30P':  1.77,
'PC':     2.15,
'SC':     2.1
}


In [3]:

class PackingDefect2:
    def __init__(self):
        pass

    def read_top(self, resname, topology_file):
        """
        Read a topology file and map each atom in the specified residue to its radius and type.

        Parameters:
        residue_name (str): Name of the residue to read from the topology file.
        topology_file (str): Path to the topology file.

        Returns:
        dict: A dictionary where keys are atom names and values are lists of [radius, type].
        """


        tails = ['C2{}'.format(i) for i in range(2, 23)] + \
                 ['C3{}'.format(i) for i in range(2, 23)] + \
                 ['H{}{}'.format(i, suffix) for i in range(2, 23) for suffix in ['R', 'S', 'X', 'Y']] + \
                 ['H16Z', 'H18T', 'H91', 'H101', 'H18Z', 'H20T']

        TGglyc = ['O11', 'O21', 'O31', 'O12', 'O22', 'O32', 'C1', 'C2', 'C3', 'C11', 'C21', 'C31', 'HA', 'HB', 'HS', 'HX', 'HY']

        COside = ['C{}'.format(i) for i in range(1, 28)] + \
                 ['H{}'.format(i) for i in range(1, 28)] + \
                 ['H{}{}'.format(i, suffix) for i in range(1, 28) for suffix in ['A', 'B']] + \
                 ['H3', 'H19C', 'H18C', 'H6', 'H14', 'H17', 'H25', 'H27C', 'H26C']

        with open(topology_file) as file:
            startread = False
            output = {}
            for line in file:
                if line.startswith('!'):
                    continue
                if line.startswith(f'RESI {resname}'):
                    startread = True
                elif startread and line.startswith('BOND'):
                    break  # Exit the loop once we reach 'BOND'
                elif startread and line.startswith('ATOM'):
                    atom_name, atom_type = line.split()[1:3]

                    acyl = -1  # Default value
                    if resname != 'TRIO':
                        if atom_name in tails:
                            acyl = 1
                    elif atom_name in TGglyc:
                        acyl = 2
                    else:
                        acyl = 3

                    output[atom_name] = [types_radii[atom_type], acyl]

        return output
    

    def defect_size(self, matrices, nbins, bin_max, prob=True):
        
        """
        Calculate the size distribution of defects in matrices.

        Parameters:
        matrices (list): List of matrices representing defects.
        nbins (int): Number of bins for the histogram.
        bin_max (float): Maximum bin value for the histogram.
        prob (bool, optional): If True, the histogram is normalized to form a probability distribution.

        Returns:
        tuple: Bin positions (midpoints) and histogram values, or (None, None) if histogram is empty.
        """
        bins = np.linspace(0, bin_max, nbins)

        defects = []
        for matrix in matrices:
            graph = self._make_graph(matrix)
            visited = set()
            for n in graph:
                if n not in visited:
                    defect_loc = self._dfs(graph, n)
                    visited.update(defect_loc)
                    defects.append(len(defect_loc))

        hist, bin_edges = np.histogram(defects, bins)
        hist = hist.astype(np.float64)
        binp = 0.5 * (bin_edges[1:] + bin_edges[:-1])

        if hist.sum() == 0:
            return None, None

        if prob:
            hist /= hist.sum()

        return binp, hist



    def _dfs(self, graph, start):
        visited, stack = set(), [start]
        while stack:
            vertex = stack.pop()
            if vertex not in visited:
                visited.add(vertex)
                stack.extend(graph[vertex] - visited)
        return visited


    def _make_graph(self, matrix):
        graph = {}
        xis, yis = matrix.shape

        for (xi, yi), value in np.ndenumerate(matrix):
            if value == 0:
                continue

            n = xi * yis + yi
            nlist = []

            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    # 8-neighbors
                    x = divmod(xi + dx, xis)[1]
                    y = divmod(yi + dy, yis)[1]
                    if matrix[x, y] == 1:
                        ndn = x * yis + y
                        nlist.append(ndn)


            graph[n] = set(nlist) - set([n])
        return graph

    

    def _make_graph(self, matrix):
        """
        Create a graph from a matrix where each non-zero element is connected to its 8-neighbors.

        Parameters:
        matrix (numpy.ndarray): A 2D array representing the matrix.

        Returns:
        dict: A dictionary representing the graph. Keys are node indices, and values are sets of neighbor indices.
        """
        graph = {}
        xis, yis = matrix.shape  # Get the dimensions of the matrix

        # Iterate over each element in the matrix
        for (xi, yi), value in np.ndenumerate(matrix):
            if value == 0:
                continue  # Skip zero values, as they are not part of the graph

            # Calculate a unique index for each node (element in the matrix)
            node_index = xi * yis + yi
            neighbor_list = []  # List to store the neighbors of the current node

            # Check all possible 8 neighbors around the current node
            for dx in [-1, 0, 1]:
                for dy in [-1, 0, 1]:
                    # Calculate the neighbor's coordinates while handling the wrapping
                    x, y = divmod(xi + dx, xis)[1], divmod(yi + dy, yis)[1]

                    # If the neighbor is non-zero and not the node itself, it's a valid neighbor
                    if matrix[x, y] == 1 and (x, y) != (xi, yi):
                        neighbor_node_index = x * yis + y  # Calculate the neighbor's unique index
                        neighbor_list.append(neighbor_node_index)  # Add it to the list

            graph[node_index] = set(neighbor_list)  # Add the node and its neighbors to the graph

        return graph


In [4]:

class PackingDefect2PMDA(ParallelAnalysisBase):
    def __init__(self, atomgroups, radii, nbins=600, bin_max=150, prefix='./', prob=True, leaflet='both'):
        """
        Initialize the PackingDefect2PMDA object.

        Parameters:
        atomgroups (list): List of atom groups to be analyzed.
        radii (dict): Dictionary mapping atom types to their radii.
        nbins (int, optional): Number of bins for histogram analysis.
        bin_max (float, optional): Maximum value for the bins.
        prefix (str, optional): Prefix for output files.
        prob (bool, optional): If True, the histogram is normalized to form a probability distribution.
        leaflet (str, optional): Specifies which leaflet to analyze ('both', 'up', or 'dw').
        """
        u = atomgroups[0].universe
        self.N = 3000  # Maximum number of defects to consider
        self.dt = u.trajectory[0].dt  # Time step of the trajectory
        self.dx = 1  # Grid spacing in x-direction
        self.dy = 1  # Grid spacing in y-direction
        self.radii = radii
        self.nbins = nbins
        self.bin_max = bin_max
        self.prefix = prefix
        self.prob = prob
        self.leaflet = leaflet
        self.all_defect_data = {}
        self.protein_atoms = u.select_atoms("protein", updating=True)  # Select protein atoms
        self.bbox_data = {"x_min": 0, "x_max": 0, "y_min": 0, "y_max": 0}  # Initialize bounding box data

        super(PackingDefect2PMDA, self).__init__(u, atomgroups)

    def _prepare(self):
        pass
    
    
    def _single_frame(self, ts, atomgroups):

        ag = atomgroups[0]  # First atomgroup in the list
        protein_atoms = ag.universe.select_atoms("protein", updating=True)  # Select protein atoms
        dim = ts.dimensions.copy()  # Copy the dimensions of the current timestep
        pbc = dim[0:3]  # Periodic boundary conditions
        print('time: {:.3f}    pbc: {:.3f} {:.3f} {:.3f}'.format(ts.time/1000, pbc[0], pbc[1], pbc[2]))

        # Adjust positions for periodic boundary conditions
        pbc_xy0 = np.array([pbc[0], pbc[1], 0])
        pbc_xyz = np.array([pbc[0], pbc[1], pbc[2]])
        aa = ag.universe.atoms
        aa.positions -= pbc_xy0 * np.floor(aa.positions / pbc_xyz)

        # Average height of phosphorus atoms in the lipid layer
        hz = np.average(ag.select_atoms('name P').positions[:, 2])

        # Calculate the bounding box for protein atoms
        protein_bbox = protein_atoms.bbox()
        x_min, y_min, _ = protein_bbox[0]
        x_max, y_max, _ = protein_bbox[1]
        # Add padding to the bounding box
        x_min -= 10  # Adding 1 nm padding (10 Å)
        x_max += 10
        y_min -= 10
        y_max += 10

        # Store updated bounding box data
        self.bbox_data["x_min"] = x_min
        self.bbox_data["x_max"] = x_max
        self.bbox_data["y_min"] = y_min
        self.bbox_data["y_max"] = y_max

        # Generate meshgrid for analysis
        xarray = np.arange(0, pbc[0], self.dx)
        yarray = np.arange(0, pbc[1], self.dy)
        xx, yy = np.meshgrid(xarray, yarray)

        # Initialize matrices for upper and lower leaflets
        M = {'up': np.zeros_like(xx), 'dw': np.zeros_like(xx)}
        Z = {'up': np.zeros_like(xx), 'dw': np.zeros_like(xx)}
        Z['up'] += hz
        Z['dw'] += hz

        # Determine the z-limits for upper and lower leaflets
        zlim = {'up': np.max(ag.positions[:, 2]), 'dw': np.min(ag.positions[:, 2])}

        # Calculate center of mass for phospholipids in each leaflet
        PL = {'up': ag.select_atoms('name P and prop z > %f' % hz).center_of_mass()[2],
              'dw': ag.select_atoms('name P and prop z < %f' % hz).center_of_mass()[2]}

        # Determine which atoms to analyze based on the specified leaflet
        atoms = {}
        if self.leaflet in ['both', 'up']:
            atoms['up'] = ag.select_atoms('prop z > %f' % (PL['up'] - 20))
        if self.leaflet in ['both', 'dw']:
            atoms['dw'] = ag.select_atoms('prop z < %f' % (PL['dw'] + 20))

        # Analyze each atom in the specified leaflets
        for l in atoms:
            for atom in atoms[l]:
                xatom, yatom, zatom = atom.position
                # Assert conditions to ensure atoms are within the correct leaflet
                if l == 'up': 
                    assert zatom > PL[l] - 20, 'check Z pos'
                if l == 'dw': 
                    assert zatom < PL[l] + 20, 'check Z pos'

                radius, acyl = self.radii[atom.resname][atom.name]
                # Calculate distances from the atom to each point in the meshgrid
                dxx = xx - xatom
                dxx -= pbc[0] * np.around(dxx/pbc[0])
                dyy = yy - yatom
                dyy -= pbc[1] * np.around(dyy/pbc[1])

                # Determine if the points in the meshgrid are within the interaction radius
                dist_meet = (np.sqrt(self.dx**2 + self.dy**2)/2 + radius)**2
                bAr = dxx ** 2 + dyy **2 < dist_meet

                # Exclude protein atoms if needed (commented out for now)
                # if atom.resname in self.protein_atoms.resnames:  
                #     continue  

                # Update matrices based on the acyl value
                if acyl == -1:
                    M[l][bAr] = acyl
                    continue
                else:
                    bAnP = M[l] >= 0
                    if l == 'up':
                        baZ = zatom > Z[l]
                    else:
                        baZ = zatom < Z[l]
                    bA = bAr & bAnP & baZ
                    M[l][bA] = acyl
                    Z[l][bA] = zatom

        return M['up'], M['dw'], PL['up']+5, PL['dw']-5, dim



    def _conclude(self):
        print("Concluding...")
        # Initialize lists to store results
        Mup = []; Mdw = []; zlimup = []; zlimdw = []; dim = []

        # Aggregate results from each processed frame
        for r in self._results:
            for rr in r:
                if rr[0] is None:
                    continue
                Mup.append(rr[0])  # Append upper leaflet matrix
                Mdw.append(rr[1])  # Append lower leaflet matrix
                zlimup.append(rr[2])  # Append upper leaflet z-limit
                zlimdw.append(rr[3])  # Append lower leaflet z-limit
                dim.append(rr[4])  # Append frame dimensions

        # Set up a new Universe for storing defect information
        N = self.N  # The maximum number of defects
        df = Universe.empty(n_atoms=N,
                            n_residues=N,
                            atom_resindex=np.arange(N),
                            residue_segindex=[0] * N,
                            trajectory=True)

        # Add topology attributes to the new Universe
        df.add_TopologyAttr('resname', ['O'] * N)
        df.add_TopologyAttr('name', ['O'] * N)
        df.add_TopologyAttr('resid', np.arange(N) + 1)

        # Initialize the trajectory data for the new Universe
        nframes = len(dim)
        fac = np.zeros((nframes, N, 3))
        df.load_new(fac, order='fac')
        df.trajectory[0].dt = self.dt  # Set the time step

        # Update the dimensions for each frame in the new Universe
        for i, ts in enumerate(df.trajectory):
            df.trajectory[i].dimensions = dim[i]

        # Define defect types
        defects = ['Deep', 'PLacyl', 'TGglyc', 'TGacyl']

        # Initialize dictionaries to store defect universe and cluster data
        defect_uni = {}
        defect_clu = {}
        for d in defects:
            defect_uni[d] = df.copy()  # Create a copy of the universe for each defect type
            defect_clu[d] = []  # Initialize an empty list for each defect type

        # Define threshold values for each defect type
        defect_thr = {'Deep': 0, 'PLacyl': 1, 'TGglyc': 2, 'TGacyl': 3}

        # Process each defect type
        for d in defects:
            for i, ts in enumerate(defect_uni[d].trajectory):
                num = 0

                # Identify defect locations
                bA = (Mup[i] == defect_thr[d])  # Get defect locations for upper leaflet
                defect_clu[d].append(bA.astype(int))
                ind = np.where(bA)
                xs, ys = ind[1], ind[0]

                # Place defects in the universe
                for x1, y1 in zip(xs, ys):
                    if num >= self.N:
                        break
                    pos = np.array([x1, y1, zlimup[i]])
                    defect_uni[d].atoms[num].position = pos
                    num += 1

                # Repeat for the lower leaflet
                bA = (Mdw[i] == defect_thr[d])
                defect_clu[d].append(bA.astype(int))
                ind = np.where(bA)
                xs, ys = ind[1], ind[0]

                for x1, y1 in zip(xs, ys):
                    if num >= self.N:
                        break
                    pos = np.array([x1, y1, zlimdw[i]])
                    defect_uni[d].atoms[num].position = pos
                    num += 1

        # Write defect data to output files
        for d in defects:
            u = defect_uni[d]
            u.trajectory[-1]
            u.atoms.write(self.prefix + d + 'rep3_TEST.gro')
            with mda.Writer(self.prefix + d + 'rep3_TEST.xtc', u.atoms.n_atoms) as W:
                for ts in u.trajectory:
                    W.write(u.atoms)

            # Write protein atom data if needed
            with mda.Writer(self.prefix + 'rep3_TEST.gro', self.protein_atoms.n_atoms) as W:
                W.write(self.protein_atoms)
            with mda.Writer(self.prefix + 'rep3_TEST.xtc', self.protein_atoms.n_atoms) as W:
                for ts in self.protein_atoms.universe.trajectory:
                    W.write(self.protein_atoms)

        # Create directories for storing the output if they don't exist
        output_base_dir = 'gro'  # Base directory for outputs
        for d in defects:
            output_dir = os.path.join(output_base_dir, self.prefix + d)
            if not os.path.exists(output_dir):
                os.makedirs(output_dir)

            # Further defect localization
            u = defect_uni[d]

            # Iterate over the trajectory and merge protein and defect data
            for i, ts in enumerate(u.trajectory):
                self.protein_atoms.universe.trajectory[i]  # Update protein positions for the current frame

                # Create a combined universe for the current frame
                combined_universe = mda.Merge(self.protein_atoms, u.atoms)

                # Update positions in the combined universe
                combined_universe.atoms.positions[len(self.protein_atoms):] = ts.positions
                combined_universe.atoms.positions[:len(self.protein_atoms)] = self.protein_atoms.positions
                combined_universe.trajectory.ts.dimensions = ts.dimensions

                # Construct file path for writing the combined universe
                output_filepath = os.path.join(output_dir, f"{d}_frame_{i}.gro")
                combined_universe.atoms.write(output_filepath)



In [5]:
if __name__ == '__main__':
    u = mda.Universe('../../trajs/6.6_2.gro')  # Initialize Universe with only the topology file

    pd = PackingDefect2()

    lipid = 'top_all36_lipid.rtf'
    TRIO  = 'TRIO.rtf'
    CHYO = 'CHYO.rtf'
    SAPI  = 'toppar_all36_lipid_inositol.str'

    radii = {'POPC': pd.read_top('POPC', lipid),
             'DOPE': pd.read_top('DOPE', lipid),
             'SAPI': pd.read_top('SAPI', SAPI),
             'TRIO': pd.read_top('TRIO', TRIO),
             'CHYO': pd.read_top('CHYO', CHYO)}

    MEMB = u.select_atoms('resname POPC DOPE SAPI TRIO CHYO')

    trajectory_files = ['../../trajs/rep3_skip10.xtc']  # Add more filenames as needed

    for traj_file in trajectory_files:
        # Load the trajectory file
        u.load_new(f'../../trajs/test/{traj_file}')

        # Create a unique directory for each trajectory
        traj_dir = f"output_{traj_file.split('.')[0]}"  # e.g., 'output_1skip'
        output_dir = os.path.join('GRO', traj_dir)
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        # Set the output prefix to the unique directory
        prefix = os.path.join(output_dir, 'results')

        # Initialize and run the analysis
        pdPMDA = PackingDefect2PMDA([MEMB], radii, prefix=prefix, leaflet='both')
        pdPMDA.run(n_jobs=2)


time: 501.000    pbc: 96.855 96.855 247.431
time: 502.000    pbc: 96.784 96.784 248.318
time: 503.000    pbc: 96.897 96.897 248.062
time: 504.000    pbc: 97.404 97.404 245.321
time: 505.000    pbc: 97.809 97.809 244.027
time: 506.000    pbc: 96.984 96.984 247.134
time: 507.000    pbc: 97.521 97.521 243.836
time: 508.000    pbc: 97.361 97.361 245.186
time: 509.000    pbc: 96.602 96.602 249.228
time: 510.000    pbc: 97.348 97.348 245.230
time: 511.000    pbc: 97.312 97.312 245.019
time: 512.000    pbc: 97.111 97.111 245.842
time: 513.000    pbc: 96.998 96.998 246.391
time: 514.000    pbc: 97.451 97.451 245.403
time: 515.000    pbc: 97.216 97.216 245.557
time: 516.000    pbc: 97.965 97.965 241.975
time: 517.000    pbc: 97.914 97.914 242.564
time: 518.000    pbc: 97.652 97.652 243.409
time: 519.000    pbc: 98.106 98.106 241.942
time: 520.000    pbc: 97.739 97.739 243.230
time: 521.000    pbc: 97.807 97.807 243.352
time: 522.000    pbc: 98.874 98.874 237.775
time: 523.000    pbc: 97.679 97.

/home/jaybraun/miniconda3/lib/python3.9/site-packages/pmda/parallel.py:440: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return np.asarray(res), np.asarray(times_io), np.asarray(


time: 873.000    pbc: 98.344 98.344 239.879
time: 874.000    pbc: 97.831 97.831 243.111
time: 875.000    pbc: 97.997 97.997 242.681
time: 876.000    pbc: 98.350 98.350 240.032
time: 877.000    pbc: 98.054 98.054 241.491
time: 878.000    pbc: 97.706 97.706 243.486
time: 879.000    pbc: 98.367 98.367 240.211
time: 880.000    pbc: 98.641 98.641 239.007
time: 881.000    pbc: 97.763 97.763 243.543
time: 882.000    pbc: 97.392 97.392 244.481
time: 883.000    pbc: 98.224 98.224 240.628
time: 884.000    pbc: 98.138 98.138 241.725
time: 885.000    pbc: 97.317 97.317 245.370
time: 886.000    pbc: 97.368 97.368 245.675
time: 887.000    pbc: 97.622 97.622 243.821
time: 888.000    pbc: 97.696 97.696 243.437
time: 889.000    pbc: 97.578 97.578 243.835
time: 890.000    pbc: 97.829 97.829 243.389
time: 891.000    pbc: 97.217 97.217 245.752
time: 892.000    pbc: 97.937 97.937 242.453
time: 893.000    pbc: 97.539 97.539 244.615
time: 894.000    pbc: 96.209 96.209 251.231
time: 895.000    pbc: 96.149 96.

time: 246.000    pbc: 95.211 95.211 256.590
time: 247.000    pbc: 95.817 95.817 252.954
time: 248.000    pbc: 95.688 95.688 253.659
time: 249.000    pbc: 95.908 95.908 252.751
time: 250.000    pbc: 95.828 95.828 253.052
time: 251.000    pbc: 96.348 96.348 250.314
time: 252.000    pbc: 96.936 96.936 247.509
time: 253.000    pbc: 96.769 96.769 248.313
time: 254.000    pbc: 96.423 96.423 250.222
time: 255.000    pbc: 97.443 97.443 245.254
time: 256.000    pbc: 95.680 95.680 253.667
time: 257.000    pbc: 95.302 95.302 255.624
time: 258.000    pbc: 95.727 95.727 253.434
time: 259.000    pbc: 96.074 96.074 252.068
time: 260.000    pbc: 96.391 96.391 250.339
time: 261.000    pbc: 96.019 96.019 252.045
time: 262.000    pbc: 95.990 95.990 251.742
time: 263.000    pbc: 95.597 95.597 254.236
time: 264.000    pbc: 95.281 95.281 256.140
time: 265.000    pbc: 96.462 96.462 249.207
time: 266.000    pbc: 96.887 96.887 247.983
time: 267.000    pbc: 96.756 96.756 248.825
time: 268.000    pbc: 97.283 97.

Concluding...


print(u.trajectory.ts, u.trajectory.time)